# Bölüm 11 — BÖLÜM 11: DERİN ÖĞRENME (DEEP LEARNING) MİMARİLERİ VE OPTİMİZASYON

**VERİ MADENCİLİĞİ VE MAKİNE ÖĞRENMESİ**  
*Python ile Temel Analitikten Büyük Veri ve Gerçek Zamanlı Sistemlere*

Bu defter, kitabın 11. bölümündeki tüm kod örneklerini içerir. Her hücrenin başlığı kitaptaki alt bölüme karşılık gelir.


In [ ]:
# Bu bölüm için gerekli paketler
!pip install -q matplotlib numpy scikit-learn tensorflow


## 11.1. Derin Ağları Eğitmenin Zorlukları ve Modern Çözümler


### Gradyan Akışı Analizi: Python Kodu

`bolum11/11_01_01_gradyan-akisi-analizi-python-kodu.py`

_Kitap: Kod 11.1_


In [ ]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────
# Gradyan Kaybolması Demonstrasyonu: Sigmoid vs ReLU
# ─────────────────────────────────────────────────────────────────────

def build_deep_network(activation, depth=10, units=64):
    """Aktivasyon fonksiyonuna göre derin ağ inşa et."""
    model = keras.Sequential(name=f"deep_{activation}")
    model.add(keras.layers.Input(shape=(20,)))
    for i in range(depth):
        model.add(keras.layers.Dense(
            units,
            activation=activation,
            kernel_initializer="glorot_uniform" if activation=="sigmoid"
                               else "he_normal",
            name=f"layer_{i+1}"
        ))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    return model

# Rastgele girdi verisi
np.random.seed(42)
X_dummy = np.random.randn(100, 20).astype("float32")
y_dummy = (np.random.rand(100) > 0.5).astype("float32")

gradient_norms = {}

for act in ["sigmoid", "relu", "elu"]:
    model = build_deep_network(act)
    model.compile(loss="binary_crossentropy", optimizer="adam")

    with tf.GradientTape() as tape:
        y_pred = model(X_dummy[:10], training=True)
        loss = keras.losses.binary_crossentropy(
            y_dummy[:10].reshape(-1,1), y_pred)
        loss = tf.reduce_mean(loss)

    # Tüm katmanlar için gradyanları hesapla
    grads = tape.gradient(loss, model.trainable_variables)
    norms = [tf.norm(g).numpy() for g in grads if g is not None]
    gradient_norms[act] = norms
    print(f"\n{act.upper():10s} aktivasyonu – Katman gradyan normları:")
    for i, n in enumerate(norms[::2]):   # Yalnızca ağırlık gradyanları
        print(f"  Katman {i+1:2d}: {n:.2e}")

# ─────────────────────────────────────────────────────────────────────
# Gradient Clipping Uygulaması
# ─────────────────────────────────────────────────────────────────────

model_clip = build_deep_network("relu")

# clipnorm: gradyan normu 1.0'ı aşarsa kırp
# clipvalue: her gradyan elemanını [-0.5, 0.5] aralığına sıkıştır
optimizer_clip = keras.optimizers.Adam(
    learning_rate=0.001,
    clipnorm=1.0          # Önerilen yöntem
    # clipvalue=0.5       # Alternatif yöntem
)

model_clip.compile(
    optimizer=optimizer_clip,
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
print("\nGradient clipping aktif Adam optimizer hazır.")

# ─────────────────────────────────────────────────────────────────────
# Residual Bağlantı: Sıfırdan Implementasyon
# ─────────────────────────────────────────────────────────────────────

def residual_block(inputs, units, name_prefix):
    """Temel residual blok: F(x) + x"""
    x = keras.layers.Dense(units, activation="relu",
                           kernel_initializer="he_normal",
                           name=f"{name_prefix}_dense1")(inputs)
    x = keras.layers.BatchNormalization(name=f"{name_prefix}_bn1")(x)
    x = keras.layers.Dense(units, activation=None,
                           kernel_initializer="he_normal",
                           name=f"{name_prefix}_dense2")(x)
    x = keras.layers.BatchNormalization(name=f"{name_prefix}_bn2")(x)
    # Skip connection: boyut eşleştirme gerekirse projection ekle
    if inputs.shape[-1] != units:
        inputs = keras.layers.Dense(units, use_bias=False,
                                    name=f"{name_prefix}_proj")(inputs)
    x = keras.layers.Add(name=f"{name_prefix}_add")([x, inputs])
    x = keras.layers.Activation("relu", name=f"{name_prefix}_relu")(x)
    return x

# Residual mimari
inputs = keras.Input(shape=(784,), name="giris")
x = keras.layers.Dense(256, activation="relu",
                        kernel_initializer="he_normal")(inputs)
x = residual_block(x, 256, "res1")
x = residual_block(x, 256, "res2")
x = residual_block(x, 128, "res3")   # boyut küçülür; projection eklenir
outputs = keras.layers.Dense(10, activation="softmax")(x)

res_model = keras.Model(inputs, outputs, name="ResidualMLP")
res_model.summary()


### LeCun Başlatması

`bolum11/11_01_02_lecun-baslatmasi.py`

_Kitap: Kod 11.2_


In [ ]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────
# Farklı Başlatma Stratejilerinin Aktivasyon Dağılımına Etkisi
# ─────────────────────────────────────────────────────────────────────

np.random.seed(42)
X_sample = np.random.randn(200, 784).astype("float32")

initializers = {
    "zeros":      keras.initializers.Zeros(),
    "random_normal(std=1)": keras.initializers.RandomNormal(stddev=1.0),
    "glorot_normal":        keras.initializers.GlorotNormal(),
    "he_normal":            keras.initializers.HeNormal(),
    "lecun_normal":         keras.initializers.LecunNormal(),
}

fig, axes = plt.subplots(1, len(initializers), figsize=(20, 4))

for ax, (name, init) in zip(axes, initializers.items()):
    # Tek katmanlı model
    layer = keras.layers.Dense(256, activation="relu",
                               kernel_initializer=init)
    out = layer(X_sample)
    ax.hist(out.numpy().flatten(), bins=50, color="steelblue", alpha=0.7)
    ax.set_title(name, fontsize=8)
    ax.set_xlabel("Aktivasyon değeri")
    mean_v = np.mean(np.abs(out.numpy()))
    ax.text(0.05, 0.95, f"Ort.|x|={mean_v:.3f}", transform=ax.transAxes,
            fontsize=8, va="top")

plt.suptitle("Başlatma Stratejisinin Aktivasyon Dağılımına Etkisi (ReLU)",
             fontweight="bold")
plt.tight_layout()
plt.savefig("initialization_comparison.png", dpi=150)
plt.show()

# ─────────────────────────────────────────────────────────────────────
# Model Karşılaştırması: Glorot vs He başlatması
# ─────────────────────────────────────────────────────────────────────

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype("float32") / 255.0
X_te = X_te.reshape(-1, 784).astype("float32") / 255.0

def build_model_with_init(init_name, activation="relu"):
    return keras.Sequential([
        keras.layers.Dense(256, activation=activation,
                           kernel_initializer=init_name, input_shape=(784,)),
        keras.layers.Dense(128, activation=activation,
                           kernel_initializer=init_name),
        keras.layers.Dense(10, activation="softmax")
    ])

configs = [
    ("glorot_normal", "sigmoid"),
    ("glorot_normal", "relu"),
    ("he_normal",     "relu"),
    ("he_normal",     "elu"),
]

for init, act in configs:
    m = build_model_with_init(init, act)
    m.compile("adam", "sparse_categorical_crossentropy", ["accuracy"])
    h = m.fit(X_tr, y_tr, epochs=10, batch_size=128,
              validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_te, y_te, verbose=0)
    print(f"Init={init:15s}  Act={act:10s}  →  Test Acc: {acc:.4f}")


### Layer Normalization ve Diğer Normalizasyon Varyantları

`bolum11/11_01_03_layer-normalization-ve-diger-normalizasyon-varya.py`

_Kitap: Kod 11.3_


In [ ]:
import tensorflow as tf
from tensorflow import keras

# ─────────────────────────────────────────────────────────────────────
# Batch Normalization Uygulaması: Doğru Kullanım
# ─────────────────────────────────────────────────────────────────────

# Yöntem 1: Orijinal kağıt sırası (Dense → BN → Activation)
model_bn = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    # BN kullandığında Dense'de bias gerekmez (BN beta parametresi bias görevi görür)
    keras.layers.Dense(300, use_bias=False, kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    keras.layers.Dense(200, use_bias=False, kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    keras.layers.Dense(100, use_bias=False, kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    keras.layers.Dense(10, activation="softmax")
], name="BN_Model")

model_bn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),  # BN ile daha yüksek LR
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# BN parametrelerini incele
bn_layer = model_bn.layers[2]   # İlk BN katmanı
print(f"BN parametreleri: {bn_layer.count_params()}")
print(f"  gamma (ölçek) : {bn_layer.gamma.shape}")
print(f"  beta (kaydırma): {bn_layer.beta.shape}")
print(f"  moving_mean    : {bn_layer.moving_mean.shape}")
print(f"  moving_variance: {bn_layer.moving_variance.shape}")

# ─────────────────────────────────────────────────────────────────────
# Layer Normalization (Transformer için)
# ─────────────────────────────────────────────────────────────────────

model_ln = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(300, kernel_initializer="he_normal"),
    keras.layers.LayerNormalization(),   # batch boyutundan bağımsız
    keras.layers.Activation("relu"),
    keras.layers.Dense(100, kernel_initializer="he_normal"),
    keras.layers.LayerNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Dense(10, activation="softmax")
], name="LN_Model")

# ─────────────────────────────────────────────────────────────────────
# BN vs BN-yok karşılaştırması
# ─────────────────────────────────────────────────────────────────────

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype("float32") / 255.0
X_te = X_te.reshape(-1, 784).astype("float32") / 255.0

for name, model in [("BN_Model", model_bn), ("LN_Model", model_ln)]:
    h = model.fit(X_tr, y_tr, epochs=10, batch_size=64,
                  validation_split=0.1, verbose=0)
    _, acc = model.evaluate(X_te, y_te, verbose=0)
    init_loss = h.history["loss"][0]
    final_loss = h.history["loss"][-1]
    print(f"{name}: Başlangıç loss={init_loss:.3f} → Final loss={final_loss:.3f}  |  Test acc={acc:.4f}")


## 11.2. Gelişmiş Optimizasyon ve Aşırı Öğrenmeyi (Overfitting) Önleme


### Adam Varyantları

`bolum11/11_02_01_adam-varyantlari.py`

_Kitap: Kod 11.4_


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────
# Optimizör Karşılaştırması: SGD, Momentum, Adam, AdamW
# ─────────────────────────────────────────────────────────────────────

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype("float32") / 255.0
X_te = X_te.reshape(-1, 784).astype("float32") / 255.0

def build_model():
    return keras.Sequential([
        keras.layers.Dense(256, activation="relu",
                           kernel_initializer="he_normal", input_shape=(784,)),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(128, activation="relu",
                           kernel_initializer="he_normal"),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(10, activation="softmax")
    ])

optimizers = {
    "SGD(lr=0.01)":           keras.optimizers.SGD(learning_rate=0.01),
    "SGD+Momentum(0.9)":      keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "SGD+Nesterov":           keras.optimizers.SGD(learning_rate=0.01, momentum=0.9,
                                                   nesterov=True),
    "RMSprop(lr=0.001)":      keras.optimizers.RMSprop(learning_rate=0.001),
    "Adam(lr=0.001)":         keras.optimizers.Adam(learning_rate=0.001),
    "AdamW(lr=0.001)":        keras.optimizers.AdamW(learning_rate=0.001,
                                                      weight_decay=0.01),
    "Nadam(lr=0.001)":        keras.optimizers.Nadam(learning_rate=0.001),
}

histories = {}
for name, opt in optimizers.items():
    model = build_model()
    model.compile(optimizer=opt,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    h = model.fit(X_tr, y_tr, epochs=15, batch_size=128,
                  validation_split=0.1, verbose=0)
    _, test_acc = model.evaluate(X_te, y_te, verbose=0)
    histories[name] = h.history["val_accuracy"]
    print(f"{name:30s} → Test Acc: {test_acc:.4f}")

# Yakınsama grafikleri
plt.figure(figsize=(12, 5))
for name, val_acc in histories.items():
    plt.plot(val_acc, label=name, lw=2)
plt.xlabel("Epoch"); plt.ylabel("Doğrulama Doğruluğu")
plt.title("Optimizör Karşılaştırması (MNIST)", fontweight="bold")
plt.legend(fontsize=8); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("optimizer_comparison.png", dpi=150)
plt.show()

# ─────────────────────────────────────────────────────────────────────
# Öğrenme Oranı Bulucu (LR Range Test)
# ─────────────────────────────────────────────────────────────────────

# LR Range Test: En uygun learning rate aralığını bul
model = build_model()

# Üstel LR zamanlayıcı: 0.00001'den 1.0'a kadar
import math
lr_finder_epochs = 30
lr_start, lr_end = 1e-5, 1.0

lr_schedule = keras.callbacks.LearningRateScheduler(
    lambda epoch: lr_start * (lr_end/lr_start) ** (epoch/lr_finder_epochs)
)

model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=lr_start),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_lr = model.fit(
    X_tr[:10000], y_tr[:10000],
    epochs=lr_finder_epochs,
    batch_size=128,
    callbacks=[lr_schedule],
    verbose=0
)

# Loss'un en hızlı düştüğü LR değeri optimal öğrenme oranına işaret eder
lrs = [lr_start * (lr_end/lr_start)**(e/lr_finder_epochs)
       for e in range(lr_finder_epochs)]
plt.figure(figsize=(8,4))
plt.semilogx(lrs, history_lr.history["loss"], lw=2)
plt.xlabel("Öğrenme Oranı (log ölçek)"); plt.ylabel("Loss")
plt.title("LR Range Test", fontweight="bold")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


### 5. Monte Carlo Dropout (MC Dropout)

`bolum11/11_02_02_monte-carlo-dropout.py`

_Kitap: Kod 11.5_


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────
# L1, L2, Dropout ve Early Stopping: Kapsamlı Karşılaştırma
# ─────────────────────────────────────────────────────────────────────

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype("float32") / 255.0
X_te = X_te.reshape(-1, 784).astype("float32") / 255.0

# Kasıtlı küçük veri seti (overfitting yaratmak için)
X_small, y_small = X_tr[:500], y_tr[:500]

def build_regularized_model(reg_type="none", dropout_rate=0.0):
    """Farklı regularization stratejileriyle model inşa et."""
    if reg_type == "l2":
        reg = keras.regularizers.L2(0.001)
    elif reg_type == "l1":
        reg = keras.regularizers.L1(0.001)
    elif reg_type == "l1_l2":
        reg = keras.regularizers.L1L2(l1=0.0001, l2=0.001)
    else:
        reg = None

    layers = [
        keras.layers.Dense(512, activation="relu",
                           kernel_initializer="he_normal",
                           kernel_regularizer=reg, input_shape=(784,)),
        keras.layers.Dense(256, activation="relu",
                           kernel_initializer="he_normal",
                           kernel_regularizer=reg),
    ]

    if dropout_rate > 0:
        layers.insert(1, keras.layers.Dropout(dropout_rate))
        layers.append(keras.layers.Dropout(dropout_rate))

    layers.append(keras.layers.Dense(10, activation="softmax"))
    return keras.Sequential(layers)

configs = [
    ("No Reg",             "none",  0.0),
    ("L2 (λ=0.001)",       "l2",    0.0),
    ("L1 (λ=0.001)",       "l1",    0.0),
    ("Dropout (p=0.3)",    "none",  0.3),
    ("Dropout (p=0.5)",    "none",  0.5),
    ("L2 + Dropout(0.3)",  "l2",    0.3),
]

results = {}
for name, reg, dr in configs:
    model = build_regularized_model(reg, dr)
    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    h = model.fit(X_small, y_small, epochs=50, batch_size=32,
                  validation_data=(X_te, y_te), verbose=0)
    tr_acc  = max(h.history["accuracy"])
    val_acc = max(h.history["val_accuracy"])
    gap = tr_acc - val_acc
    results[name] = (tr_acc, val_acc, gap)
    print(f"{name:25s}  Train: {tr_acc:.3f}  Val: {val_acc:.3f}  Gap: {gap:.3f}")

# Karşılaştırma: En küçük gap = en az overfitting
print("\n→ Gap küçüldükçe overfitting azalıyor.")

# ─────────────────────────────────────────────────────────────────────
# Early Stopping: Detaylı Kullanım
# ─────────────────────────────────────────────────────────────────────

model_es = build_regularized_model("l2", 0.3)
model_es.compile("adam", "sparse_categorical_crossentropy", ["accuracy"])

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        min_delta=0.001,
        restore_best_weights=True,   # ← KRİTİK: En iyi ağırlıklara dön
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
]

history = model_es.fit(
    X_tr, y_tr,
    epochs=100,
    batch_size=64,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

print(f"\nEğitim {len(history.history['loss'])} epoch'ta durdu.")
print(f"En iyi val_loss: {min(history.history['val_loss']):.4f}")

# ─────────────────────────────────────────────────────────────────────
# Veri Artırma (Data Augmentation) ile Eğitim
# ─────────────────────────────────────────────────────────────────────

(X_tr_img, y_tr_img), (X_te_img, y_te_img) = keras.datasets.cifar10.load_data()
X_tr_img = X_tr_img.astype("float32") / 255.0
X_te_img = X_te_img.astype("float32") / 255.0

# Keras Preprocessing katmanları ile gerçek zamanlı augmentation
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),          # ±10% döndürme
    keras.layers.RandomZoom(0.1),              # ±10% zoom
    keras.layers.RandomTranslation(0.1, 0.1), # Kaydırma
    keras.layers.RandomBrightness(0.2),        # Parlaklık değişimi
    keras.layers.RandomContrast(0.1),          # Kontrast değişimi
], name="augmentation")

# CNN modeli (augmentation dahil)
cnn_inputs = keras.Input(shape=(32, 32, 3))
x = data_augmentation(cnn_inputs, training=True)  # Yalnızca eğitimde aktif
x = keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = keras.layers.MaxPooling2D()(x)
x = keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = keras.layers.MaxPooling2D()(x)
x = keras.layers.Flatten()(x)
x = keras.layers.Dropout(0.5)(x)
cnn_outputs = keras.layers.Dense(10, activation="softmax")(x)

cnn_aug = keras.Model(cnn_inputs, cnn_outputs)
cnn_aug.compile("adam", "sparse_categorical_crossentropy", ["accuracy"])
print("CNN + Augmentation modeli hazır:", cnn_aug.input_shape)

# ─────────────────────────────────────────────────────────────────────
# Monte Carlo Dropout: Belirsizlik Tahmini
# ─────────────────────────────────────────────────────────────────────

# MC Dropout: Çıkarım sırasında Dropout aktif tut
model_mc = keras.Sequential([
    keras.layers.Dense(256, activation="relu", input_shape=(784,)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation="softmax")
])
model_mc.compile("adam", "sparse_categorical_crossentropy", ["accuracy"])
model_mc.fit(X_tr, y_tr, epochs=5, verbose=0)

# MC Dropout ile belirsizlik tahmini
test_sample = X_te[:1]
n_samples = 100

# training=True ile dropout aktif
mc_predictions = np.stack([
    model_mc(test_sample, training=True).numpy() for _ in range(n_samples)
])

mean_pred = mc_predictions.mean(axis=0)
std_pred  = mc_predictions.std(axis=0)

predicted_class = np.argmax(mean_pred)
confidence = mean_pred[0, predicted_class]
uncertainty = std_pred[0, predicted_class]

print(f"\nMC Dropout ({n_samples} örnek):")
print(f"  Tahmin edilen sınıf: {predicted_class}")
print(f"  Ortalama güven    : {confidence:.4f}")
print(f"  Belirsizlik (std) : {uncertainty:.4f}")


## 11.3. Evrişimsel Sinir Ağları (Convolutional Neural Networks — CNN)


### Python Uygulaması: CIFAR-10 CNN Modeli

`bolum11/11_03_04_python-uygulamasi-cifar-10-cnn-modeli.py`

_Kitap: Kod 11.6_


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ============================================================
# 1. VERİ YÜKLEME VE ÖN İŞLEME
# ============================================================
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Normalizasyon: [0,255] → [0,1]
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

# CIFAR-10 sınıf etiketleri
sinif_adlari = ['uçak','otomobil','kuş','kedi','geyik',
                'köpek','kurbağa','at','gemi','kamyon']

print(f'Eğitim: {X_train.shape}, Test: {X_test.shape}')
print(f'Görüntü boyutu: {X_train.shape[1:]}  (32×32×3 RGB)')

# ============================================================
# 2. CNN MİMARİSİ: VGG-STYLE KÜÇÜK MODEL
# ============================================================
def cnn_modeli_olustur():
    model = keras.Sequential([
        # --- BLOK 1 ---
        layers.Conv2D(32, (3,3), padding='same', activation='relu',
                      input_shape=(32,32,3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # --- BLOK 2 ---
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.30),

        # --- BLOK 3 ---
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),  # Flatten yerine GAP
        layers.Dropout(0.40),

        # --- SINIFLANDIRICI ---
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.50),
        layers.Dense(10, activation='softmax'),
    ], name='VGGStyle_CIFAR10')
    return model

model = cnn_modeli_olustur()
model.summary()

# ============================================================
# 3. VERİ ARTIRIMI (DATA AUGMENTATION)
# ============================================================
datagen = ImageDataGenerator(
    rotation_range=15,         # ±15° döndürme
    width_shift_range=0.1,     # Yatay kaydırma
    height_shift_range=0.1,    # Dikey kaydırma
    horizontal_flip=True,      # Yatay çevirme
    zoom_range=0.1,            # Yakınlaştırma
    fill_mode='nearest'
)
datagen.fit(X_train)

# ============================================================
# 4. DERLEME VE EĞİTİM
# ============================================================
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.001, decay_steps=50*390)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('cifar10_best.h5', save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6),
]

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=64),
    epochs=50,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# 5. RESNET-STYLE SKIP CONNECTION (FUNCTIONAL API)
# ============================================================
def residual_blok(x, filters, stride=1):
    """Temel ResNet artık bloku — F(x) + x"""
    shortcut = x

    # Ana yol
    x = layers.Conv2D(filters, 3, stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)

    # Boyut eşleştirme (stride > 1 ise shortcut boyutu değişir)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, stride)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Skip connection: F(x) + x
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

def kucuk_resnet():
    giris = keras.Input(shape=(32, 32, 3))
    x = layers.Conv2D(32, 3, padding='same')(giris)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = residual_blok(x, 32)
    x = residual_blok(x, 64, stride=2)
    x = residual_blok(x, 64)
    x = residual_blok(x, 128, stride=2)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(10, activation='softmax')(x)
    return keras.Model(giris, x, name='KucukResNet')

resnet = kucuk_resnet()
resnet.summary()
print(f'\nResNet parametre sayısı: {resnet.count_params():,}')


## 11.4. Tekrarlayan Sinir Ağları (Recurrent Neural Networks — RNN) ve Sıralı Veriler


### Python Uygulaması: LSTM ile Zaman Serisi Tahmini

`bolum11/11_04_03_python-uygulamasi-lstm-ile-zaman-serisi-tahmini.py`

_Kitap: Kod 11.7_


In [ ]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# ============================================================
# 1. SİNÜS DALGA TAHMINI (TEMEL ZAMAN SERİSİ)
# ============================================================
np.random.seed(42)
t = np.linspace(0, 100, 1000)
# Gürültülü sinüs sinyali — gerçekçi zaman serisi simülasyonu
zaman_serisi = np.sin(0.1 * t) + 0.5 * np.sin(0.5 * t) + 0.1 * np.random.randn(1000)

def dizi_olustur(veri, n_adim):
    """Kayan pencere ile giriş-çıktı çiftleri oluşturur"""
    X, y = [], []
    for i in range(len(veri) - n_adim):
        X.append(veri[i:i + n_adim])
        y.append(veri[i + n_adim])
    return np.array(X), np.array(y)

n_adim = 30  # Son 30 adımı kullanarak bir sonraki değeri tahmin et
X, y = dizi_olustur(zaman_serisi, n_adim)

# Normalizasyon
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
y_scaled = scaler.fit_transform(y.reshape(-1,1)).ravel()

# Train/test split
split = int(0.8 * len(X_scaled))
X_train = X_scaled[:split].reshape(-1, n_adim, 1)
X_test  = X_scaled[split:].reshape(-1, n_adim, 1)
y_train = y_scaled[:split]
y_test  = y_scaled[split:]

print(f'Eğitim: {X_train.shape}, Test: {X_test.shape}')

# ============================================================
# 2. LSTM MODELİ
# ============================================================
lstm_model = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(n_adim, 1)),
    layers.Dropout(0.2),
    layers.LSTM(32, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='linear')  # Regresyon çıktısı
], name='LSTM_ZamanSerisi')

lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)
lstm_model.summary()

history = lstm_model.fit(
    X_train, y_train,
    epochs=50, batch_size=32,
    validation_split=0.2,
    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=0
)

test_loss, test_mae = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f'Test MSE: {test_loss:.6f}, Test MAE: {test_mae:.6f}')

# ============================================================
# 3. GRU vs LSTM KARŞILAŞTIRMASI
# ============================================================
def model_olustur(tur, n_adim):
    model = keras.Sequential(name=tur)
    if tur == 'SimpleRNN':
        model.add(layers.SimpleRNN(64, return_sequences=True, input_shape=(n_adim,1)))
        model.add(layers.SimpleRNN(32))
    elif tur == 'LSTM':
        model.add(layers.LSTM(64, return_sequences=True, input_shape=(n_adim,1)))
        model.add(layers.LSTM(32))
    elif tur == 'GRU':
        model.add(layers.GRU(64, return_sequences=True, input_shape=(n_adim,1)))
        model.add(layers.GRU(32))
    model.add(layers.Dense(16, activation='relu'))
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

print('\n=== RNN / LSTM / GRU Karşılaştırması ===')
print(f'{"Model":<12} {"Parametre":>12} {"Test MAE":>10}')
print('-' * 37)

for tur in ['SimpleRNN', 'GRU', 'LSTM']:
    m = model_olustur(tur, n_adim)
    m.fit(X_train, y_train, epochs=30, batch_size=32,
          validation_split=0.1, verbose=0)
    _, mae = m.evaluate(X_test, y_test, verbose=0)
    params = m.count_params()
    print(f'{tur:<12} {params:>12,} {mae:>10.6f}')

# ============================================================
# 4. DUYGU ANALİZİ: Many-to-One LSTM
# ============================================================
print('\n=== Duygu Analizi: LSTM Many-to-One ===\n')

max_kelime = 10000
max_uzunluk = 200
embedding_dim = 64

# IMDB veri seti
(X_imdb_train, y_imdb_train), (X_imdb_test, y_imdb_test) = \
    keras.datasets.imdb.load_data(num_words=max_kelime)

X_imdb_train = keras.preprocessing.sequence.pad_sequences(
    X_imdb_train, maxlen=max_uzunluk, padding='post')
X_imdb_test = keras.preprocessing.sequence.pad_sequences(
    X_imdb_test, maxlen=max_uzunluk, padding='post')

duygu_model = keras.Sequential([
    # Embedding: Kelime indekslerini yoğun vektörlere dönüştürür
    layers.Embedding(max_kelime, embedding_dim, input_length=max_uzunluk),
    layers.Dropout(0.2),
    # İki katmanlı Bidirectional LSTM
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Dropout(0.3),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')  # İkili sınıflandırma
], name='Bidirectional_LSTM_Duygu')

duygu_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
duygu_model.summary()

# (Eğitim satırı — gerçekte çalıştırılabilir)
# history = duygu_model.fit(X_imdb_train, y_imdb_train,
#     epochs=5, batch_size=128, validation_split=0.2)
# Beklenen Test Doğruluğu: ~%87-89


## 11.5. Transfer Öğrenimi (Transfer Learning)


### Python Uygulaması I: EfficientNetB4 Aşamalı Fine-Tuning

`bolum11/11_05_02_python-uygulamasi-i-efficientnetb4-asamali-fine.py`

_Kitap: Kod 11.8_


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB4, ResNet50V2
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_prep
from tensorflow.keras.applications.resnet_v2 import preprocess_input as res_prep

# ============================================================
# 1. ÖZELLIK ÇIKARIMI — ResNet50V2
# ============================================================
def ozellik_cikarici_model(n_sinif, base='resnet50v2', img_size=224):
    """
    Önceden eğitilmiş modelin tüm ağırlıkları dondurulur.
    Yalnızca yeni sınıflandırma başlığı eğitilir.
    """
    input_shape = (img_size, img_size, 3)

    if base == 'resnet50v2':
        base_model = ResNet50V2(weights='imagenet', include_top=False,
                                input_shape=input_shape)
    else:
        base_model = EfficientNetB4(weights='imagenet', include_top=False,
                                    input_shape=input_shape)

    # TÜM BASE MODEL DONDUR
    base_model.trainable = False

    # Giriş + preprocessing model içinde
    giris = keras.Input(shape=input_shape, name='goruntu_girisi')
    x = res_prep(giris)                          # Model-spesifik normalizasyon
    x = base_model(x, training=False)            # training=False: BN inference mod
    x = layers.GlobalAveragePooling2D()(x)       # Flatten yerine GAP
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    cikis = layers.Dense(n_sinif, activation='softmax')(x)

    model = keras.Model(giris, cikis, name='FeatureExtraction_ResNet50V2')

    frozen  = sum(1 for l in base_model.layers if not l.trainable)
    toplam  = base_model.count_params()
    egitim  = model.count_params() - toplam
    print(f'Donuk parametre : {toplam:>12,}  ({frozen} katman)')
    print(f'Eğitilen param. : {egitim:>12,}  ({100*egitim/(toplam+egitim):.1f}% toplam)')
    return model

# ============================================================
# 2. AŞAMALI İNCE AYARLAMA — EfficientNetB4
# ============================================================
def asamali_fine_tuning(n_sinif, img_size=224):
    """
    Üç aşamalı progressive fine-tuning:
    Aşama 1: Yalnızca baş (head) eğitimi — 10 epoch
    Aşama 2: Son 30 katman açıldı — 10 epoch (LR 10x küçültüldü)
    Aşama 3: Son 60 katman açıldı — 20 epoch (LR 100x küçültüldü)
    """
    input_shape = (img_size, img_size, 3)
    base = EfficientNetB4(weights='imagenet', include_top=False,
                          input_shape=input_shape)
    base.trainable = False

    giris = keras.Input(shape=input_shape)
    x = eff_prep(giris)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    cikis = layers.Dense(n_sinif, activation='softmax')(x)
    model = keras.Model(giris, cikis, name='ProgressiveFineTune_EfficientNetB4')

    callbacks_genel = [
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7),
    ]

    # --- AŞAMA 1: Sadece baş eğitimi ---
    print('\n=== AŞAMA 1: Baş katman eğitimi (LR=1e-3) ===')
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    # history1 = model.fit(X_train, y_train, epochs=10, callbacks=callbacks_genel, ...)

    # --- AŞAMA 2: Son 30 katman açıldı ---
    print('\n=== AŞAMA 2: Son 30 katman fine-tuning (LR=1e-4) ===')
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    # BatchNorm katmanlarını daima inference modunda tut
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    donuk_2 = sum(1 for l in base.layers if not l.trainable)
    print(f'  Dondurulmuş: {donuk_2} / {len(base.layers)} katman')

    model.compile(optimizer=keras.optimizers.Adam(1e-4),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    # history2 = model.fit(X_train, y_train, epochs=10, callbacks=callbacks_genel, ...)

    # --- AŞAMA 3: Son 60 katman açıldı ---
    print('\n=== AŞAMA 3: Son 60 katman fine-tuning (LR=5e-6) ===')
    for layer in base.layers[-60:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True

    donuk_3 = sum(1 for l in base.layers if not l.trainable)
    print(f'  Dondurulmuş: {donuk_3} / {len(base.layers)} katman')

    model.compile(optimizer=keras.optimizers.Adam(5e-6),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    # history3 = model.fit(X_train, y_train, epochs=20, callbacks=callbacks_genel, ...)

    model.summary()
    return model

print('=== Transfer Öğrenimi Model Kurulumu ===')
print('\n--- Özellik Çıkarımı Modeli (ResNet50V2) ---')
fe_model = ozellik_cikarici_model(n_sinif=5)
print('\n--- Aşamalı Fine-Tuning Modeli (EfficientNetB4) ---')
ft_model = asamali_fine_tuning(n_sinif=5)


### Python Uygulaması II: Veri Artırımı ile Tam Transfer Öğrenim Pipeline'ı

`bolum11/11_05_02_python-uygulamasi-ii-veri-artirimi-ile-tam-trans.py`

_Kitap: Kod 11.9_


In [ ]:
import numpy as np
# ============================================================
# TAM TRANSFER ÖĞRENİMİ PIPELINE'I
# Gerçek dünya projesinde tüm adımlar birlikte
# ============================================================
from tensorflow.keras.applications import MobileNetV3Large

def transfer_pipeline(veri_dizini, n_sinif, img_size=224, batch_size=32):
    """
    Gerçek bir transfer öğrenimi pipeline'ı:
    1. Veri yükleme + augmentation
    2. Özellik çıkarımı ile ısınma
    3. Aşamalı fine-tuning
    """

    # ---- VERİ ARTIRIMI (Keras Preprocessing Layers — model içinde) ----
    veri_artirimi = keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.10),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
        layers.RandomBrightness(factor=0.10),
        layers.RandomTranslation(0.05, 0.05),
    ], name='veri_artirimi')

    # ---- MODEL KURULUMU ----
    giris = keras.Input(shape=(img_size, img_size, 3))
    x = veri_artirimi(giris)          # Sadece training=True'da aktif

    base = MobileNetV3Large(weights='imagenet', include_top=False,
                             input_tensor=x)
    base.trainable = False            # Başlangıçta dondur

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    cikis = layers.Dense(n_sinif, activation='softmax')(x)
    model = keras.Model(giris, cikis, name='MobileNetV3_Pipeline')

    # ---- AŞAMA 1: ISıNMA (Warm-up) ----
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    print(f'Aşama 1 — Eğitilen param: {sum(np.prod(w.shape) for w in model.trainable_weights):,}')

    # Gerçek veriyle eğitim:
    # train_ds = tf.keras.utils.image_dataset_from_directory(
    #     veri_dizini, validation_split=0.2, subset='training',
    #     seed=42, image_size=(img_size,img_size), batch_size=batch_size)
    # val_ds = tf.keras.utils.image_dataset_from_directory(
    #     veri_dizini, validation_split=0.2, subset='validation',
    #     seed=42, image_size=(img_size,img_size), batch_size=batch_size)
    # Performans için: train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)

    # ---- AŞAMA 2: FİNE-TUNİNG ----
    base.trainable = True
    # BatchNorm frozen
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    print(f'Aşama 2 — Eğitilen param: {sum(np.prod(w.shape) for w in model.trainable_weights):,}')
    return model

model_full = transfer_pipeline('/data/siniflar', n_sinif=10)
print('\nTransfer pipeline hazır!')


## 11.6. Üretici ve Temsili Modeller (Kısa Bir Bakış)


### Python Uygulaması: Otokodlayıcı ile Anomali Tespiti

`bolum11/11_06_01_python-uygulamasi-otokodlayici-ile-anomali-tespi.py`

_Kitap: Kod 11.10_


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score, classification_report

# ============================================================
# 1. VERİ HAZIRLAMA — MNIST Anomali Senaryosu
# ============================================================
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

# SENARYO: Yalnızca '0' rakamları normal; diğerleri anomali
X_normal_train = X_train[y_train == 0]
y_anomali_test = (y_test != 0).astype(int)   # 0=normal, 1=anomali

print(f'Normal eğitim: {len(X_normal_train)} örnek')
print(f'Test seti: {len(y_test)} örnek | Anomali oranı: {y_anomali_test.mean():.1%}')

# ============================================================
# 2. CNN OTOKODLAYıCı MİMARİSİ
# ============================================================
def otokodlayici_olustur(latent_dim=32):
    # --- KODLAYICI ---
    enc_in = keras.Input(shape=(28, 28, 1), name='encoder_giris')
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(enc_in)
    x = layers.MaxPooling2D(2)(x)              # → 14×14×32
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2)(x)              # → 7×7×64
    x = layers.Flatten()(x)                    # → 3136
    gizli = layers.Dense(latent_dim, name='gizli_kod')(x)  # DARBOĞAZ
    encoder = keras.Model(enc_in, gizli, name='Encoder')

    # --- KOD ÇÖZÜCÜ ---
    dec_in = keras.Input(shape=(latent_dim,), name='decoder_giris')
    x = layers.Dense(7*7*64, activation='relu')(dec_in)
    x = layers.Reshape((7, 7, 64))(x)
    x = layers.Conv2DTranspose(64, 3, activation='relu', padding='same')(x)
    x = layers.UpSampling2D(2)(x)              # → 14×14×64
    x = layers.Conv2DTranspose(32, 3, activation='relu', padding='same')(x)
    x = layers.UpSampling2D(2)(x)              # → 28×28×32
    yeniden = layers.Conv2DTranspose(1, 3, activation='sigmoid',
                                      padding='same', name='cikis')(x)
    decoder = keras.Model(dec_in, yeniden, name='Decoder')

    # --- TAM OTOKODLAYıCı ---
    inp = encoder.input
    out = decoder(encoder(inp))
    oto = keras.Model(inp, out, name='Otokodlayici')
    return encoder, decoder, oto

encoder, decoder, oto = otokodlayici_olustur(latent_dim=32)
oto.summary()

# ============================================================
# 3. YALNIZCA NORMAL VERİYLE EĞİTİM
# ============================================================
oto.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')

X_tr = X_normal_train.reshape(-1, 28, 28, 1)
X_te = X_test.reshape(-1, 28, 28, 1)

history = oto.fit(
    X_tr, X_tr,                     # Girdi = Hedef (self-supervised)
    epochs=30, batch_size=128,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(patience=5,
                                              restore_best_weights=True)],
    verbose=0
)
print(f'Son eğitim kaybı: {history.history["loss"][-1]:.6f}')

# ============================================================
# 4. ANOMALİ TESPİTİ VE DEĞERLENDİRME
# ============================================================
X_hat = oto.predict(X_te, verbose=0)
recon_hatalari = np.mean((X_te - X_hat)**2, axis=(1,2,3))

auc = roc_auc_score(y_anomali_test, recon_hatalari)
print(f'\nAnomali Tespiti AUC-ROC: {auc:.4f}')

# Eşik: eğitim hatalarının 95. yüzdeliği
egitim_hatalari = np.mean(
    (X_tr - oto.predict(X_tr, verbose=0))**2, axis=(1,2,3))
esik = np.percentile(egitim_hatalari, 95)
y_tahmin = (recon_hatalari > esik).astype(int)

print(f'Eşik (θ = %95. yüzdelik): {esik:.6f}')
print('\n' + classification_report(y_anomali_test, y_tahmin,
      target_names=['Normal', 'Anomali']))

normal_ort  = recon_hatalari[y_anomali_test == 0].mean()
anomali_ort = recon_hatalari[y_anomali_test == 1].mean()
print(f'Normal ortalama hata :  {normal_ort:.6f}')
print(f'Anomali ortalama hata:  {anomali_ort:.6f}  ({anomali_ort/normal_ort:.1f}× daha yüksek)')


### Python Uygulaması: DCGAN ile MNIST Görüntü Üretimi

`bolum11/11_06_02_python-uygulamasi-dcgan-ile-mnist-goruntu-uretim.py`

_Kitap: Kod 11.11, Kod 11.12_


In [ ]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ============================================================
# 1. VERİ HAZIRLAMA
# ============================================================
(X_train, _), (_, _) = keras.datasets.mnist.load_data()
X_train = X_train.astype('float32')
# GAN için [-1, 1] normalizasyonu (Generator tanh çıkışıyla uyumlu)
X_train = (X_train - 127.5) / 127.5
X_train = X_train.reshape(-1, 28, 28, 1)
print(f'Veri: {X_train.shape}  Aralık: [{X_train.min():.1f}, {X_train.max():.1f}]')

# ============================================================
# 2. ÜRETİCİ (GENERATOR)
# ============================================================
def uretici_olustur(latent_dim=100):
    """
    z ~ N(0,I) → 28×28×1 sahte görüntü [-1, 1]
    Transposed convolution ile kademeli boyut büyütme
    """
    model = keras.Sequential([
        # Gürültü vektörünü 7×7×256'ya genişlet
        layers.Dense(7 * 7 * 256, input_dim=latent_dim),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Reshape((7, 7, 256)),

        # 7×7×256 → 14×14×128
        layers.Conv2DTranspose(128, 4, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        # 14×14×128 → 28×28×64
        layers.Conv2DTranspose(64, 4, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        # 28×28×64 → 28×28×1 (son katman tanh ile [-1,1])
        layers.Conv2DTranspose(1, 4, strides=1, padding='same', activation='tanh'),
    ], name='Generator')
    return model

# ============================================================
# 3. AYIRT EDİCİ (DISCRIMINATOR)
# ============================================================
def ayirt_edici_olustur():
    """
    28×28×1 görüntü → gerçeklik skoru [0,1]
    NOT: D'de BatchNorm KULLANILMAZ (eğitim kararlılığı için)
    """
    model = keras.Sequential([
        # 28×28×1 → 14×14×64
        layers.Conv2D(64, 4, strides=2, padding='same', input_shape=(28,28,1)),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),

        # 14×14×64 → 7×7×128
        layers.Conv2D(128, 4, strides=2, padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1, activation='sigmoid'),  # Gerçek mi sahte mi?
    ], name='Discriminator')
    return model

# ============================================================
# 4. GAN SINIFI — ÖZEL EĞİTİM DÖNGÜSÜ
# ============================================================
class DCGAN(keras.Model):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.latent_dim = latent_dim
        self.G = uretici_olustur(latent_dim)
        self.D = ayirt_edici_olustur()

    def compile(self, d_opt, g_opt, loss_fn):
        super().compile()
        self.d_opt = d_opt
        self.g_opt = g_opt
        self.loss_fn = loss_fn
        self.d_loss_m = keras.metrics.Mean('d_loss')
        self.g_loss_m = keras.metrics.Mean('g_loss')

    def train_step(self, gercek):
        bs = tf.shape(gercek)[0]

        # ---- ADIM 1: D Güncelle ----
        z = tf.random.normal([bs, self.latent_dim])
        sahte = self.G(z, training=False)
        tum   = tf.concat([gercek, sahte], axis=0)
        # Label smoothing: 1.0→0.9, 0.0→0.1
        etiket = tf.concat([
            tf.ones((bs,1)) * 0.9,   # Gerçek: smoothed
            tf.zeros((bs,1)) + 0.1,  # Sahte:  smoothed
        ], axis=0)

        with tf.GradientTape() as tape:
            tahmin = self.D(tum, training=True)
            d_loss = self.loss_fn(etiket, tahmin)
        grads = tape.gradient(d_loss, self.D.trainable_variables)
        self.d_opt.apply_gradients(zip(grads, self.D.trainable_variables))

        # ---- ADIM 2: G Güncelle ----
        z = tf.random.normal([bs, self.latent_dim])
        aldatici = tf.ones((bs, 1))  # G, D'nin 'gerçek' demesini istiyor

        with tf.GradientTape() as tape:
            sahte2 = self.G(z, training=True)
            pred   = self.D(sahte2, training=False)  # D donduruldu
            g_loss = self.loss_fn(aldatici, pred)
        grads = tape.gradient(g_loss, self.G.trainable_variables)
        self.g_opt.apply_gradients(zip(grads, self.G.trainable_variables))

        self.d_loss_m.update_state(d_loss)
        self.g_loss_m.update_state(g_loss)
        return {'d_loss': self.d_loss_m.result(),
                'g_loss': self.g_loss_m.result()}

# ============================================================
# 5. EĞİTİM VE GÖRÜNTÜ ÜRETME
# ============================================================
LATENT_DIM = 100
gan = DCGAN(latent_dim=LATENT_DIM)
gan.compile(

    g_opt  = keras.optimizers.Adam(2e-4, beta_1=0.5),
    loss_fn= keras.losses.BinaryCrossentropy(),
)

print(f'Generator parametresi    : {gan.G.count_params():,}')
print(f'Discriminator parametresi: {gan.D.count_params():,}')

# Eğitim (gerçek GPU ortamında çalıştırılır):
# history = gan.fit(X_train, epochs=50, batch_size=128)

# Eğitim sonrası yeni görüntü üretme:
# gurultu = np.random.normal(0, 1, (25, LATENT_DIM))
# uretilen = gan.G.predict(gurultu)
# uretilen = (uretilen * 127.5 + 127.5).astype(np.uint8)  # [-1,1] → [0,255]
print('\nGAN hazır. Eğitim tamamlandığında G.predict() ile görüntü üretilir.')
